In [7]:
import tensorflow as tf

encoder = tf.keras.Sequential([tf.keras.layers.Dense(2)])
decoder = tf.keras.Sequential([tf.keras.layers.Dense(3)])
autoencoder = tf.keras.Sequential([encoder, decoder])
optimizer = tf.keras.optimizers.SGD(learning_rate=0.5)
autoencoder.compile(loss="mse", optimizer=optimizer)

In [8]:
X_train = [...] # generate a 3D dataset, like in Chapter 8
history = autoencoder.fit(X_train, X_train, epochs=500, verbose=False)
codings = encoder.predict(X_train)

ValueError: Unrecognized data type: x=[Ellipsis] (of type <class 'list'>)

In [ ]:
stacked_encoder = tf.keras.Sequential([
 tf.keras.layers.Flatten(),
 tf.keras.layers.Dense(100, activation="relu"),
 tf.keras.layers.Dense(30, activation="relu"),
])
stacked_decoder = tf.keras.Sequential([
 tf.keras.layers.Dense(100, activation="relu"),
 tf.keras.layers.Dense(28 * 28),
 tf.keras.layers.Reshape([28, 28])
])
stacked_ae = tf.keras.Sequential([stacked_encoder, stacked_decoder])
stacked_ae.compile(loss="mse", optimizer="nadam")
history = stacked_ae.fit(X_train, X_train, epochs=20,
 validation_data=(X_valid, X_valid))

NameError: name 'X_valid' is not defined

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
def plot_reconstructions(model, images=X_valid, n_images=5):
    reconstructions = np.clip(model.predict(images[:n_images]), 0, 1)
    fig = plt.figure(figsize=(n_images * 1.5, 3))
    for image_index in range(n_images):
        plt.subplot(2, n_images, 1 + image_index)
        plt.imshow(images[image_index], cmap="binary")
        plt.axis("off")
        plt.subplot(2, n_images, 1 + n_images + image_index)
        plt.imshow(reconstructions[image_index], cmap="binary")
        plt.axis("off")
    plot_reconstructions(stacked_ae)
    plt.show()

NameError: name 'X_valid' is not defined

In [ ]:
from sklearn.manifold import TSNE
X_valid_compressed = stacked_encoder.predict(X_valid)
tsne = TSNE(init="pca", learning_rate="auto", random_state=42)
X_valid_2D = tsne.fit_transform(X_valid_compressed)

NameError: name 'X_valid' is not defined

In [ ]:
plt.scatter(X_valid_2D[:, 0], X_valid_2D[:, 1], c=y_valid, s=10, cmap="tab10")
plt.show()


NameError: name 'X_valid_2D' is not defined

In [9]:
class DenseTranspose(tf.keras.layers.Layer):
    
    def __init__(self, dense, activation=None, **kwargs):
        super().__init__(**kwargs)
        self.dense = dense
        self.activation = tf.keras.activations.get(activation)
    
    def build(self, batch_input_shape):
        self.biases = self.add_weight(name="bias",
        shape=self.dense.input_shape[-1],
        initializer="zeros")
        super().build(batch_input_shape)
    
    def call(self, inputs):
        Z = tf.matmul(inputs, self.dense.weights[0], transpose_b=True)
        return self.activation(Z + self.biases)


In [11]:
dense_1 = tf.keras.layers.Dense(100, activation="relu")
dense_2 = tf.keras.layers.Dense(30, activation="relu")
tied_encoder = tf.keras.Sequential([
 tf.keras.layers.Flatten(),
 dense_1,
 dense_2
])
tied_decoder = tf.keras.Sequential([
 DenseTranspose(dense_2, activation="relu"),
 DenseTranspose(dense_1),
 tf.keras.layers.Reshape([28, 28])
])
tied_ae = tf.keras.Sequential([tied_encoder, tied_decoder])

In [12]:
conv_encoder = tf.keras.Sequential([
 tf.keras.layers.Reshape([28, 28, 1]),
 tf.keras.layers.Conv2D(16, 3, padding="same", activation="relu"),
 tf.keras.layers.MaxPool2D(pool_size=2), # output: 14 × 14 x 16
 tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
 tf.keras.layers.MaxPool2D(pool_size=2), # output: 7 × 7 x 32
 tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
 tf.keras.layers.MaxPool2D(pool_size=2), # output: 3 × 3 x 64
 tf.keras.layers.Conv2D(30, 3, padding="same", activation="relu"),
 tf.keras.layers.GlobalAvgPool2D() # output: 30
])

conv_decoder = tf.keras.Sequential([
 tf.keras.layers.Dense(3 * 3 * 16),
 tf.keras.layers.Reshape((3, 3, 16)),
 tf.keras.layers.Conv2DTranspose(32, 3, strides=2, activation="relu"),
 tf.keras.layers.Conv2DTranspose(16, 3, strides=2, padding="same",
 activation="relu"),
 tf.keras.layers.Conv2DTranspose(1, 3, strides=2, padding="same"),
 tf.keras.layers.Reshape([28, 28])
])

In [13]:
dropout_encoder = tf.keras.Sequential([
 tf.keras.layers.Flatten(),
 tf.keras.layers.Dropout(0.5),
 tf.keras.layers.Dense(100, activation="relu"),
 tf.keras.layers.Dense(30, activation="relu")
])
dropout_decoder = tf.keras.Sequential([
 tf.keras.layers.Dense(100, activation="relu"),
 tf.keras.layers.Dense(28 * 28),
 tf.keras.layers.Reshape([28, 28])
])
dropout_ae = tf.keras.Sequential([dropout_encoder, dropout_decoder])


In [14]:
sparse_l1_encoder = tf.keras.Sequential([
 tf.keras.layers.Flatten(),
 tf.keras.layers.Dense(100, activation="relu"),
 tf.keras.layers.Dense(300, activation="sigmoid"),
 tf.keras.layers.ActivityRegularization(l1=1e-4)
])
sparse_l1_decoder = tf.keras.Sequential([
 tf.keras.layers.Dense(100, activation="relu"),
 tf.keras.layers.Dense(28 * 28),
 tf.keras.layers.Reshape([28, 28])
])
sparse_l1_ae = tf.keras.Sequential([sparse_l1_encoder, sparse_l1_decoder])

In [15]:
codings_size = 30

Dense = tf.keras.layers.Dense
generator = tf.keras.Sequential([
 Dense(100, activation="relu", kernel_initializer="he_normal"),
 Dense(150, activation="relu", kernel_initializer="he_normal"),
 Dense(28 * 28, activation="sigmoid"),
 tf.keras.layers.Reshape([28, 28])
])

discriminator = tf.keras.Sequential([
 tf.keras.layers.Flatten(),
 Dense(150, activation="relu", kernel_initializer="he_normal"),
 Dense(100, activation="relu", kernel_initializer="he_normal"),
 Dense(1, activation="sigmoid")
])
gan = tf.keras.Sequential([generator, discriminator])

In [18]:
discriminator.compile(loss="binary_crossentropy", optimizer="rmsprop")
discriminator.trainable = False
gan.compile(loss="binary_crossentropy", optimizer="rmsprop")

In [19]:
batch_size = 32
dataset = tf.data.Dataset.from_tensor_slices(X_train).shuffle(buffer_size=1000)
dataset = dataset.batch(batch_size, drop_remainder=True).prefetch(1)

ValueError: Attempt to convert a value (Ellipsis) with an unsupported type (<class 'ellipsis'>) to a Tensor.

In [20]:
def train_gan(gan, dataset, batch_size, codings_size, n_epochs):
    generator, discriminator = gan.layers
    for epoch in range(n_epochs):
        for X_batch in dataset:
            # phase 1 - training the discriminator
            noise = tf.random.normal(shape=[batch_size, codings_size])
            generated_images = generator(noise)
            X_fake_and_real = tf.concat([generated_images, X_batch], axis=0)
            y1 = tf.constant([[0.]] * batch_size + [[1.]] * batch_size)
            discriminator.train_on_batch(X_fake_and_real, y1)
            # phase 2 - training the generator
            noise = tf.random.normal(shape=[batch_size, codings_size])
            y2 = tf.constant([[1.]] * batch_size)
            gan.train_on_batch(noise, y2)
train_gan(gan, dataset, batch_size, codings_size, n_epochs=50)


NameError: name 'dataset' is not defined

In [21]:
codings = tf.random.normal(shape=[batch_size, codings_size])
generated_images = generator.predict(codings)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 609ms/step


In [22]:
codings_size = 100

generator = tf.keras.Sequential([
 tf.keras.layers.Dense(7 * 7 * 128),
 tf.keras.layers.Reshape([7, 7, 128]),
 tf.keras.layers.BatchNormalization(),
 tf.keras.layers.Conv2DTranspose(64, kernel_size=5, strides=2,
 padding="same", activation="relu"),
 tf.keras.layers.BatchNormalization(),
 tf.keras.layers.Conv2DTranspose(1, kernel_size=5, strides=2,
 padding="same", activation="tanh"),
])

discriminator = tf.keras.Sequential([
 tf.keras.layers.Conv2D(64, kernel_size=5, strides=2, padding="same",
 activation=tf.keras.layers.LeakyReLU(0.2)),
 tf.keras.layers.Dropout(0.4),
 tf.keras.layers.Conv2D(128, kernel_size=5, strides=2, padding="same",
 activation=tf.keras.layers.LeakyReLU(0.2)),
 tf.keras.layers.Dropout(0.4),
 tf.keras.layers.Flatten(),
 tf.keras.layers.Dense(1, activation="sigmoid")
])

gan = tf.keras.Sequential([generator, discriminator])

In [23]:
X_train_dcgan = X_train.reshape(-1, 28, 28, 1) * 2. - 1. # reshape and rescale

AttributeError: 'list' object has no attribute 'reshape'

In [24]:
def prepare_batch(X):
    X = tf.cast(X[..., tf.newaxis], tf.float32) * 2 - 1 # scale from –1 to +1
    X_shape = tf.shape(X)
    t = tf.random.uniform([X_shape[0]], minval=1, maxval=T + 1, dtype=tf.int32)
    alpha_cm = tf.gather(alpha_cumprod, t)
    alpha_cm = tf.reshape(alpha_cm, [X_shape[0]] + [1] * (len(X_shape) - 1))
    noise = tf.random.normal(X_shape)
    return {
    "X_noisy": alpha_cm ** 0.5 * X + (1 - alpha_cm) ** 0.5 * noise,
    "time": t,
    }, noise


In [26]:
def prepare_batch(X):
    X = tf.cast(X[..., tf.newaxis], tf.float32) * 2 - 1 # scale from –1 to +1
    X_shape = tf.shape(X)
    t = tf.random.uniform([X_shape[0]], minval=1, maxval=T + 1, dtype=tf.int32)
    alpha_cm = tf.gather(alpha_cumprod, t)
    alpha_cm = tf.reshape(alpha_cm, [X_shape[0]] + [1] * (len(X_shape) - 1))
    noise = tf.random.normal(X_shape)
    return {
    "X_noisy": alpha_cm ** 0.5 * X + (1 - alpha_cm) ** 0.5 * noise,
    "time": t,
    }, noise

In [27]:
def prepare_dataset(X, batch_size=32, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices(X)
    if shuffle:
        ds = ds.shuffle(buffer_size=10_000)
    return ds.batch(batch_size).map(prepare_batch).prefetch(1)
train_set = prepare_dataset(X_train, batch_size=32, shuffle=True)
valid_set = prepare_dataset(X_valid, batch_size=32)


ValueError: Attempt to convert a value (Ellipsis) with an unsupported type (<class 'ellipsis'>) to a Tensor.

In [28]:
def build_diffusion_model():
    X_noisy = tf.keras.layers.Input(shape=[28, 28, 1], name="X_noisy")
    time_input = tf.keras.layers.Input(shape=[], dtype=tf.int32, name="time")
    [...] # build the model based on the noisy images and the time steps
    outputs = [...] # predict the noise (same shape as the input images)
    return tf.keras.Model(inputs=[X_noisy, time_input], outputs=[outputs])
